# Text-only GRPO · Qwen3.5-2B → arithmetic reasoning

Load [mlx-community/Qwen3.5-2B-4bit](https://huggingface.co/mlx-community/Qwen3.5-2B-4bit)
and train with GRPO with GRPO on [mlx-community/simple-grpo](https://huggingface.co/datasets/mlx-community/simple-grpo).
The dataset contains addition questions and verifiable numeric answers.
No images, supervised warm-up, teacher responses, or learned reward model are used.

Inspired by [DeepSeek-R1-Zero](https://arxiv.org/html/2501.12948v1#S2.SS2), use
correctness and format rewards, asking the model to generate
<code>&lt;think&gt;...&lt;/think&gt;&lt;answer&gt;...&lt;/answer&gt;</code>.
This is a small arithmetic GRPO example. The selected checkpoint is already
post-trained; this notebook adds no SFT warm-up. It does not reproduce
DeepSeek's base-model starting point, training scale, or benchmark results.

## Setup

Use an Apple Silicon Mac and install into your notebook kernel environment:

~~~sh
python -m pip install -e "/absolute/path/to/mlx-vlm[train]" jupyterlab ipykernel pandas wandb
~~~

After installing or upgrading packages, use **Restart Kernel** before running the cells. The first load downloads the
already-quantized 4-bit MLX checkpoint selected by MODEL_ID; subsequent runs
reuse the Hugging Face cache. No local conversion step is needed. Qwen's vision tower stays frozen and receives no inputs.

Training and validation metrics are sent to W&B. Authenticate with `wandb login` before training;
set WANDB_PROJECT to None in the settings cell to disable reporting.

### If imports fail after upgrading Hugging Face Hub

For an error importing **as_extended_path** from huggingface_hub.utils._paths,
restart the notebook kernel first. A running kernel can retain old modules while
other files on disk have already been upgraded. Re-running the import cell does
not clear those modules.

If the same error occurs after a full kernel restart, repair the package in the
active notebook environment using this separate cell:

~~~python
%pip install --force-reinstall --no-cache-dir --no-deps "huggingface-hub==2.2.0"
~~~

Then restart the kernel again. Use the notebook's %pip so the installation targets
the selected kernel rather than a different terminal environment.


In [ ]:
import json
import re
from dataclasses import asdict, replace
from decimal import Decimal, InvalidOperation
from pathlib import Path

import mlx.core as mx
import mlx.optimizers as optim
import pandas as pd
from datasets import load_dataset

from mlx_vlm import load
from mlx_vlm.trainer import (
    GRPOTrainingArgs,
    Trainer,
    WandBCallback,
    prepare_model_for_training,
)
from mlx_vlm.trainer.vlm.grpo.trainer import generate_grpo


## 1. Settings

MODEL_ID selects the existing 4-bit MLX checkpoint. Adapters live separately in OUTPUT_DIR. The Hub login used by the notebook
must be able to read the selected repository.
Batch size counts questions, group size counts sampled answers per question,
and iters counts microsteps. Four accumulation steps make one optimizer update.

Beta 0.04 uses a separate frozen copy of the same quantized base for KL
regularization. Set beta to 0.0 to skip reference allocation and scoring.


In [ ]:
MODEL_ID = "mlx-community/Qwen3.5-2B-4bit"
DATASET_ID = "mlx-community/simple-grpo"
OUTPUT_DIR = Path("adapters/vlm/grpo_text_simple")
WANDB_PROJECT = "mlx-vlm-grpo-text"  # Set to None to disable W&B reporting.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
MAX_TEST_SAMPLES = 16
SEED = 42
TRAIN_TYPE = "lora"  # "dora" also works; full training needs more memory.
CHECKPOINT_PATH = None
LORA_RANK = 8
LORA_ALPHA = 16

FORMAT_INSTRUCTION = (
    "A conversation between User and Assistant. The user asks a question, and the "
    "Assistant solves it. The assistant first thinks about the reasoning process in "
    "the mind and then provides the user with the answer. The reasoning process and "
    "answer are enclosed within <think> </think> and <answer> </answer> tags, "
    "respectively, i.e., <think> reasoning process here </think> <answer> answer here "
    "</answer>."
)
# GRPO variants share this trainer; choose their loss normalization below:
#   "grpo": mean over tokens per completion, then mean over completions.
#   "bnpo": divide the summed loss by total completion tokens.
#   "dr_grpo": divide by completion count * max_completion_length.
# Rewards always use group standard-deviation scaling, including "dr_grpo";
# this option selects the denominator, not every change in the Dr. GRPO method.
# "sequence" importance sampling gives GSPO-style ratios; epsilon_high
# sets a DAPO-style upper clipping width (defaults to epsilon when None).
# Each rollout is used once: ratios start at 1 and clipping normally stays 0.
# No multi-epoch PPO or DAPO dynamic resampling is implemented.
# beta controls reference-model KL; beta=0 skips reference scoring.
# advantage_epsilon stabilizes reward scaling; min_p also controls sampling.
R1_PROMPT_SOURCE = "https://arxiv.org/html/2501.12948v1#S2.T1"

args = GRPOTrainingArgs(
    iters=100,
    batch_size=1,
    group_size=6,
    max_completion_length=128,
    max_seq_length=1024,
    temperature=0.5,
    top_p=0.95,
    top_k=20,
    beta=0.04,
    epsilon=0.2,
    grpo_loss_type="grpo",  # also "bnpo" or "dr_grpo"
    importance_sampling_level="token",  # or "sequence" (GSPO-style)
    end_answer_token="</answer>",
    gradient_accumulation_steps=1,
    grad_checkpoint=True,
    logit_chunk_size=32,
    cache_size=32,
    compile=False,
    val_batches=5,
    steps_per_report=10,
    steps_per_eval=50,
    steps_per_save=50,
    adapter_file=str(OUTPUT_DIR / "weights.safetensors"),
)
mx.random.seed(SEED)
args.validate()


## 2. Keep the supplied data splits separate

simple-grpo supplies train (400 rows), valid (50), and test (50).
Training uses only train; progress checks use valid; the final cell uses test.
Targets are retained only as reward metadata. Mapping does not inject the
answer or an example solution into the prompt.


In [ ]:
raw_dataset = load_dataset(DATASET_ID)
required_splits = {"train", "valid", "test"}
if not required_splits.issubset(raw_dataset):
    raise ValueError(f"Expected train/valid/test splits, got {list(raw_dataset)}.")


def limit_rows(rows, maximum):
    if maximum is None:
        return rows
    if maximum < 1:
        raise ValueError("Sample limits must be positive or None.")
    return rows.select(range(min(maximum, len(rows))))


def mapp_func(sample):
    question, answer = sample.get("prompt"), sample.get("answer")
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Each question must contain a non-empty prompt string.")
    if answer is None or not str(answer).strip():
        raise ValueError("Every arithmetic question needs a numeric answer.")
    try:
        number = Decimal(str(answer).strip())
    except InvalidOperation as error:
        raise ValueError("Each answer must be a finite number.") from error
    if not number.is_finite():
        raise ValueError("Each answer must be a finite number.")
    prompt = f"{FORMAT_INSTRUCTION}\nUser: {question}\nAssistant:"
    return {"prompt": prompt, "question": question, "answer": str(answer).strip()}


def prepare_rows(rows, maximum):
    rows = limit_rows(rows, maximum)
    return rows.map(mapp_func, remove_columns=rows.column_names)


train_rows = prepare_rows(raw_dataset["train"], MAX_TRAIN_SAMPLES)
eval_rows = prepare_rows(raw_dataset["valid"], MAX_EVAL_SAMPLES)
test_rows = prepare_rows(raw_dataset["test"], MAX_TEST_SAMPLES)
assert min(len(train_rows), len(eval_rows), len(test_rows)) >= args.batch_size
print({"train": len(train_rows), "validation": len(eval_rows), "test": len(test_rows)})
print(train_rows[0])


## 3. Rule-based rewards

Accuracy is the main signal (weight 1.0). A smaller format reward (weight 0.1)
encourages one non-empty reasoning block followed by one numeric answer block.
The reward verifies the final answer, not the quality of the reasoning text.

The correctness checker accepts a single answer block or a bare numeric
response, so the policy can receive correctness feedback before it learns the
format. It rejects duplicate answer blocks, incomplete tags, and nonnumeric
answers; Decimal equality accepts equivalent forms such as 139 and 139.0.
This parser and the weights are choices for this tutorial.


In [ ]:
NUMBER_PATTERN = re.compile(r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?")
ANSWER_PATTERN = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.DOTALL)
FORMAT_PATTERN = re.compile(
    r"\s*<think>(?P<reasoning>.*?)</think>\s*<answer>(?P<answer>.*?)</answer>\s*",
    re.DOTALL,
)
TAGS = ("<think>", "</think>", "<answer>", "</answer>")


def extract_answer(completion):
    blocks = ANSWER_PATTERN.findall(completion)
    if blocks:
        if (
            len(blocks) != 1
            or completion.count("<answer>") != 1
            or completion.count("</answer>") != 1
        ):
            return None
        text = blocks[0].strip()
    else:
        text = completion.strip()
    if NUMBER_PATTERN.fullmatch(text) is None:
        return None
    try:
        number = Decimal(text)
    except InvalidOperation:
        return None
    return number if number.is_finite() else None


def correctness_reward(*, completions, answer, **kwargs):
    scores = []
    for completion, target in zip(completions, answer):
        predicted = extract_answer(completion)
        scores.append(
            float(predicted is not None and predicted == Decimal(str(target).strip()))
        )
    return scores


def reasoning_format_reward(*, completions, **kwargs):
    scores = []
    for completion in completions:
        match = FORMAT_PATTERN.fullmatch(completion)
        valid = (
            match is not None
            and all(completion.count(tag) == 1 for tag in TAGS)
            and bool(match.group("reasoning").strip())
            and NUMBER_PATTERN.fullmatch(match.group("answer").strip()) is not None
        )
        scores.append(float(valid))
    return scores


reward_funcs = [correctness_reward, reasoning_format_reward]
reward_weights = [1.0, 0.1]
print(
    correctness_reward(
        completions=["<think>56 + 83 = 139.</think><answer>139</answer>"],
        answer=["139"],
    )
)
print(
    reasoning_format_reward(
        completions=["<think>56 + 83 = 139.</think><answer>139</answer>"]
    )
)


## 4. Load the existing 4-bit checkpoint

The policy and reference load mlx-community/Qwen3.5-2B-4bit directly;
only the policy receives language adapters. No local conversion is needed.
The reference preserves the starting policy for KL regularization.

This community checkpoint was converted from post-trained Qwen3.5-2B.
The rule-based reasoning rewards are inspired by R1-Zero, while the starting
checkpoint differs from that paper's pretrained-only initialization.


In [ ]:
model, processor = load(MODEL_ID)
reference_model = load(MODEL_ID)[0] if args.beta > 0 else None
model = prepare_model_for_training(
    model,
    train_type=TRAIN_TYPE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    checkpoint_path=CHECKPOINT_PATH,
)


### Use the mapped prompt directly

mapp_func builds the published DeepSeek instruction, the question, and the
Assistant: suffix in the prompt column. The trainer tokenizes that string directly
with dataset_config={"apply_chat_template": False}; no chat-template override is
needed. The answer stays in its own column for rewards and never enters the
model prompt. The question column preserves the original text for inspection.


In [ ]:
training_tokenizer = getattr(processor, "tokenizer", processor)

optimizer = optim.AdamW(
    learning_rate=1e-5,
    weight_decay=0.0,
)

trainer = Trainer(
    model,
    args,
    optimizer=optimizer,
    algorithm="grpo",
    processing_class=processor,
    dataset_config={"apply_chat_template": False},
    train_dataset=train_rows,
    eval_dataset=eval_rows,
    reference_model=reference_model,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
    seed=SEED,
)
prepared_sample = trainer.train_dataset[0]
print(training_tokenizer.decode(prepared_sample["inputs"]["input_ids"].tolist()))
assert "pixel_values" not in prepared_sample["inputs"]


## 5. Baseline and training

Validation samples fresh groups. Compare correctness and format reward means,
not only GRPO loss. Groups with identical rewards have zero policy advantage;
monitor grouped_reward_std. If the policy receives almost no reward,
inspect its responses and adjust sampling, token budget, or question difficulty.

The memory optimizations match the vision GRPO path: CPU reward normalization,
completion-only vocabulary projection, synchronized gradients per completion,
and transformer checkpointing. KL uses the frozen starting checkpoint.


In [ ]:
baseline = trainer.evaluate()
print(
    {
        key: value
        for key, value in baseline.items()
        if "reward" in key or key == "val_loss"
    }
)
wandb_callback = None
if WANDB_PROJECT:
    wandb_callback = WandBCallback(
        project_name=WANDB_PROJECT,
        log_dir=str(OUTPUT_DIR),
        config={
            "model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "seed": SEED,
            "train_type": TRAIN_TYPE,
            "reward_weights": reward_weights,
            **asdict(args),
        },
    )
    trainer.training_callback = wandb_callback

try:
    metrics = trainer.train()
finally:
    if wandb_callback is not None:
        wandb_callback.finish()
assert mx.isfinite(mx.array(metrics["train_loss"])).item()
reports = pd.DataFrame(trainer.log_history)
reports.tail(5).reindex(
    columns=[
        "iteration",
        "train_loss",
        "train_reward_0_correctness_reward",
        "train_reward_1_reasoning_format_reward",
        "train_grouped_reward_std",
        "train_kl",
        "trained_tokens",
        "peak_memory",
    ]
)


## 6. Compare validation and inspect a generated group

Correctness reward is the sampled numeric accuracy; reasoning format reward
is the fraction of responses matching the requested tag structure. Use more
validation batches for a steadier estimate. These figures measure this small
addition dataset, not general reasoning performance.


In [ ]:
validation = trainer.evaluate()
comparison = pd.DataFrame([baseline, validation], index=["before GRPO", "after GRPO"])
comparison.reindex(
    columns=[
        "val_reward_0_correctness_reward",
        "val_reward_1_reasoning_format_reward",
        "val_reward_mean",
        "val_grouped_reward_std",
        "val_kl",
    ]
).round(4)


In [ ]:
sample = trainer.eval_dataset[0]
completion_ids, completion_texts, _ = generate_grpo(
    model,
    training_tokenizer,
    [sample],
    args,
)
print("Question:", sample["metadata"]["question"])
print("Target:", sample["answer"])
for index, text in enumerate(completion_texts):
    accuracy = correctness_reward(completions=[text], answer=[sample["answer"]])[0]
    format_score = reasoning_format_reward(completions=[text])[0]
    print(f"Sample {index + 1} (correct={accuracy}, format={format_score}):\n{text}")


## 7. Final held-out test

Use the supplied test partition only after training/settings decisions.
This cell evaluates all selected test rows rather than the short validation budget.


In [ ]:
test_args = replace(args, val_batches=-1)
test_trainer = Trainer(
    model,
    test_args,
    algorithm="grpo",
    processing_class=processor,
    dataset_config={"apply_chat_template": False},
    eval_dataset=test_rows,
    reference_model=reference_model,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
)
test_metrics = test_trainer.evaluate()
pd.DataFrame([test_metrics]).reindex(
    columns=[
        "val_reward_0_correctness_reward",
        "val_reward_1_reasoning_format_reward",
        "val_reward_mean",
        "val_kl",
    ]
).round(4)


## 8. Save adapters and the plain prompt template

The loop saves final adapter weights automatically. Keep the adapter metadata,
plain prompt template, and run settings together. Resuming adapters starts a
fresh optimizer; no processor or tokenizer chat-template changes are required.


In [ ]:
trainer.save_adapter()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prompt_template = f"{FORMAT_INSTRUCTION}\nUser: {{question}}\nAssistant:"
(OUTPUT_DIR / "prompt_template.txt").write_text(prompt_template, encoding="utf-8")
(OUTPUT_DIR / "run_config.json").write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "base_model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "format_instruction": FORMAT_INSTRUCTION,
            "prompt_template": prompt_template,
            "dataset_config": {"apply_chat_template": False},
            "prompt_source": R1_PROMPT_SOURCE,
            "train_type": TRAIN_TYPE,
            "reward_weights": reward_weights,
            "training_args": asdict(args),
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
print(Path(args.adapter_file).resolve())


Reload later and build prompts with the same plain template:

~~~python
model, processor = load(MODEL_ID)
model = prepare_model_for_training(
    model, train_type=TRAIN_TYPE, checkpoint_path=args.adapter_file,
)
template = (OUTPUT_DIR / "prompt_template.txt").read_text(encoding="utf-8")
prompt = template.format(question="What is 56 + 83?")
model.eval()
~~~

Pass mapped prompt rows to Trainer with dataset_config={"apply_chat_template": False}.
See the [training guide](../../docs/training.md) and the
[vision GRPO notebook](grpo_vision.ipynb) for image-conditioned training.
